In [1]:
import anndata as ad
import seaborn as sns
import numpy as np
import scanpy as sc
import pandas as pd
import gc
from pathlib import Path
import matplotlib.pyplot as plt

In [2]:
# Make dictionary for modules and dataframes
module_dfs = {}
modules = {}

### Control

In [3]:
# Loading data for the condition
condition = "Control"

In [4]:
# Load modules dataframe
module_dfs[condition] = pd.read_csv(
    f"/data/scRNA/Hammond/Modules/Hammond_adult_hdWGCNA_HVG2K_{condition}_modules.csv",
    index_col = "Unnamed: 0")
module_dfs[condition].head()

,gene_name,module,color,kME_NORM-NoRP-M1,kME_NORM-NoRP-M2,kME_grey,kME_NORM-NoRP-M3,kME_NORM-NoRP-M4
Cd74,Cd74,NORM-NoRP-M1,brown,0.150583,0.419821,0.018544,0.175201,-0.040476
Apoe,Apoe,NORM-NoRP-M1,brown,0.152592,0.300385,0.036027,0.236322,-0.015315
H2.Ab1,H2.Ab1,NORM-NoRP-M2,blue,0.103400,0.393140,0.000632,0.121813,-0.045858
H2.Aa,H2.Aa,NORM-NoRP-M1,brown,0.124929,0.394893,-0.008771,0.112483,-0.048529
H2.Eb1,H2.Eb1,NORM-NoRP-M1,brown,0.120687,0.390966,-0.000478,0.119566,-0.055707


In [5]:
# Rename the modules in the modules dataframe
module_dfs[condition]['module'] = module_dfs[condition]['module'].str.replace("NORM-NoRP", "MG-Control")
module_dfs[condition].rename(columns=lambda col: col.replace("NORM-NoRP", "MG-Control") if "NORM-NoRP" in col else col, inplace=True)

# Getting names of modules
modules[condition] = module_dfs[condition]["module"].unique().tolist()
modules[condition].remove("grey")
modules[condition]

['MG-Control-M1', 'MG-Control-M2', 'MG-Control-M3', 'MG-Control-M4']

In [7]:
# Loading data for the condition
condition = "Saline"

In [8]:
# Load modules dataframe
module_dfs[condition] = pd.read_csv(
    f"/data/scRNA/Hammond/Modules/Hammond_adult_hdWGCNA_HVG2K_{condition}_modules.csv",
    index_col = "Unnamed: 0")
module_dfs[condition].head()

,gene_name,module,color,kME_SALINE-NoRP-M1,kME_grey,kME_SALINE-NoRP-M2,kME_SALINE-NoRP-M3
Apoe,Apoe,SALINE-NoRP-M1,blue,0.328732,0.441214,0.320913,0.270716
Ccl4,Ccl4,grey,grey,0.162866,0.414139,0.149515,0.129828
Ccl12,Ccl12,SALINE-NoRP-M1,blue,0.543086,0.105978,0.105050,0.050860
Ccl3,Ccl3,SALINE-NoRP-M1,blue,0.093986,0.320780,0.147746,0.129622
Lpl,Lpl,SALINE-NoRP-M1,blue,0.235353,0.472862,0.204790,0.199218


In [9]:
# Rename the modules in the modules dataframe
module_dfs[condition]['module'] = module_dfs[condition]['module'].str.replace("SALINE-NoRP", "MG-Saline")
module_dfs[condition].rename(columns=lambda col: col.replace("SALINE-NoRP", "MG-Saline") if "SALINE-NoRP" in col else col, inplace=True)

# Getting names of modules
modules[condition] = module_dfs[condition]["module"].unique().tolist()
modules[condition].remove("grey")
modules[condition]

['MG-Saline-M1', 'MG-Saline-M2', 'MG-Saline-M3']

### LC

In [10]:
# Loading data for the condition
condition = "LC"

In [11]:
# Load modules dataframe
module_dfs[condition] = pd.read_csv(
    f"/data/scRNA/Hammond/Modules/Hammond_adult_hdWGCNA_HVG2K_{condition}_modules.csv",
    index_col = "Unnamed: 0")
module_dfs[condition].head()

,gene_name,module,color,kME_INJURY-NORP-M1,kME_INJURY-NORP-M2,kME_grey,kME_INJURY-NORP-M3
Ccl4,Ccl4,INJURY-NORP-M1,turquoise,0.449157,0.202290,0.093461,-0.192942
Cxcl10,Cxcl10,INJURY-NORP-M1,turquoise,0.448920,0.072827,0.053286,-0.153673
Ccl5,Ccl5,INJURY-NORP-M1,turquoise,0.306227,0.095109,0.015581,-0.194044
Ccl3,Ccl3,INJURY-NORP-M1,turquoise,0.413146,0.229298,0.095060,-0.208930
Ccl12,Ccl12,INJURY-NORP-M1,turquoise,0.496430,0.123100,0.075157,-0.224386


In [12]:
# Rename the modules in the modules dataframe
module_dfs[condition]['module'] = module_dfs[condition]['module'].str.replace("INJURY-NORP", "MG-LC")
module_dfs[condition].rename(columns=lambda col: col.replace("INJURY-NORP", "MG-LC") if "INJURY-NORP" in col else col, inplace=True)

# Getting names of modules
modules[condition] = module_dfs[condition]["module"].unique().tolist()
modules[condition].remove("grey")
modules[condition]

['MG-LC-M1', 'MG-LC-M2', 'MG-LC-M3']

In [14]:
# Create a dictionary to store genes for each module
module_genes_dict = {}

for cond, df in module_dfs.items():
    for module in modules[cond]:
        # Filter genes belonging to the current module
        module_genes = df[df['module'] == module].copy()
        # Sort genes by kME value for the current module
        kME_col = [col for col in module_genes.columns if f"kME_{module}" in col]
        if kME_col:
            module_genes = module_genes.sort_values(by=kME_col[0], ascending=False)
        # Store genes in the dictionary
        module_genes_dict[module] = module_genes['gene_name'].tolist()

# Convert the dictionary to a DataFrame
combined_df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in module_genes_dict.items()]))

combined_df

,MG-Control-M1,MG-Control-M2,MG-Control-M3,MG-Control-M4,MG-Saline-M1,MG-Saline-M2,MG-Saline-M3,MG-LC-M1,MG-LC-M2,MG-LC-M3
0,Gfm2,Ifitm3,Tmsb4x,Bmp2k,Ifitm3,Gm11808,Ftl1,Cd52,Ftl1,Jun
1,Gm26917,Ifi204,Fth1,Bhlhe41,Ifit3,Gm8730,Gapdh,Ifi27l2a,Gapdh,Btg2
2,Gm8730,Oasl2,B2m,Notch2,Isg15,Gm2000,Fau,Cst7,Gm10076,Tmem119
3,Gm20721,Ifi27l2a,Eef1a1,Slc39a1,Ifi204,Gm10116,Cd63,Ifitm3,Uba52,Crybb1
4,Gm10116,Ifit3,Fau,Ptafr,Oasl2,Gm10073,Lag3,Irf7,Serf2,Slc2a5
...,...,...,...,...,...,...,...,...,...,...
643,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ints7,NaN,NaN
644,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Pde7a,NaN,NaN
645,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Nsun3,NaN,NaN
646,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Eri2,NaN,NaN


In [15]:
combined_df.to_csv(
    "/data/scRNA/Hammond/Modules/Hammond_adult_hdWGCNA_HVG2K_combined_modules.csv")